In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
import xgboost as xgb
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [2]:
df = pd.read_parquet("../data/aircraft engine/PM_train.parquet")

In [3]:
x = df.drop(['cycle','RUL', 'max'], axis=1)
y = df['RUL']
x_train,x_test,y_train,y_test = train_test_split(x,y, test_size=0.21, random_state=42)
scaler = MinMaxScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [4]:
import json
with open('note.json', 'r') as file:
    info = json.loads(file.read())

In [5]:
info

{'gb_model': {'r2_score': 0.8581007137699551,
  'best_params': {'subsample': 1.0,
   'n_estimators': 2000,
   'min_samples_split': 2,
   'min_samples_leaf': 3,
   'max_depth': 7,
   'loss': 'squared_error',
   'learning_rate': 0.003,
   'criterion': 'squared_error',
   'alpha': 0.5}},
 'xgb_model': {'r2_score': 0.8799066543579102,
  'best_params': {'colsample_bytree': 1.0,
   'gamma': 1,
   'grow_policy': 'depthwise',
   'learning_rate': 0.024233782342509082,
   'max_depth': 7,
   'n_estimators': 1000,
   'reg_alpha': 0,
   'reg_lambda': 10,
   'subsample': 0.8}}}

In [6]:
xgb_model = xgb.XGBRegressor(colsample_bytree=1, gamma=1, learning_rate=0.0242337, max_depth=7, n_estimators=1000, reg_alpha=0, reg_lambda=10, subsample=0.8, random_state=42)

In [7]:
xgb_model.fit(x_train_scaled, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None, colsample_bytree=1,
             device=None, early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=1, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.0242337, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [8]:
y_pred = xgb_model.predict(x_test_scaled)

In [9]:
r2_score(y_test, y_pred)

0.8796504139900208

In [10]:
test_df = pd.read_csv("../data/aircraft engine/PM_test.csv")

In [11]:
test = test_df.drop('cycle', axis=1)
test = scaler.transform(test)

In [12]:
truth_df = pd.read_csv("../data/aircraft engine/PM_truth.csv")

In [13]:
pred = xgb_model.predict(test)

In [14]:
pred_df = pd.DataFrame(pred)

In [15]:
pred_df = pd.concat([pred_df, test_df['id']], axis=1)

In [16]:
pred_df

,0,id
0,158.093826,1
1,151.490952,1
2,127.498810,1
3,139.926254,1
4,172.244293,1
...,...,...
13091,28.683743,100
13092,26.357550,100
13093,33.114307,100
13094,24.523359,100


In [17]:
pred_df = pred_df.groupby('id')[0].min()
pred_df = pd.DataFrame(pred_df)

In [18]:
pred_df

,0
id,
1,121.579750
2,86.102577
3,40.715061
4,84.567665
5,76.779678
...,...
96,128.750870
97,60.150307
98,35.418823


In [19]:
truth_df

,id,cycle
0,1,112
1,2,98
2,3,69
3,4,82
4,5,91
...,...,...
95,96,137
96,97,82
97,98,59
98,99,117


In [20]:
pred_df.columns = ['cycle']

In [21]:
diff = pred_df['cycle'].values - truth_df['cycle'].values

In [22]:
print((len(np.where((diff>10) | (diff<-10))[0]),diff[np.where(diff>10)].sum()))

(69, 921.3925704956055)


In [26]:
len(np.where((diff>5) | (diff<-5))[0])

80

In [23]:
r2_score(truth_df['cycle'].values,pred_df['cycle'].values)

0.3675646185874939